# SAIFEN — 02 · Limpeza e padronização

Aplica `cleaner.clean()` e produz o DataFrame canônico que alimenta
o KDE e o `summary.json`.

Schema de saída (colunas):

| coluna         | tipo       | origem                                  |
|----------------|------------|------------------------------------------|
| lat, lng       | float64    | LATITUDE / LONGITUDE                     |
| crime_type     | str        | RUBRICA → furto / roubo / outros         |
| period         | str        | HORA_OCORRENCIA → manha/tarde/noite/madrugada |
| occurred_at    | datetime64 | DATA_OCORRENCIA_BO                       |
| neighborhood   | str        | BAIRRO (UPPER)                           |
| city           | str        | CIDADE (UPPER)                           |
| street         | str        | LOGRADOURO                               |
| police_unit    | str        | NOME_DELEGACIA                           |
| phone_brand    | str        | MARCA_OBJETO                             |
| bo_number      | str        | NUM_BO                                   |
| source_file    | str        | arquivo de origem                        |

In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from saifen_pipeline import config, loader, cleaner, exporter

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Carregar todos os .xlsx de `data/raw/`

`load_all_raw()` concatena automaticamente quando novos arquivos
(`CelularesSubtraidos_2027.xlsx`, etc) forem adicionados ao MVP.

In [ ]:
df_raw = loader.load_all_raw()
print(f"Linhas brutas: {len(df_raw):,}")
print(f"Arquivos: {df_raw.attrs.get('source_files')}")

## 2. Aplicar `cleaner.clean()`

Bbox padrão = `config.SP_BBOX` (SP capital). Para análises estaduais, passe
`bbox=None`.

In [ ]:
df = cleaner.clean(df_raw, bbox=config.SP_BBOX, drop_duplicate_bo=True)

print(f"Linhas após limpeza: {len(df):,}")
print(f"Taxa de descarte:    {df.attrs['drop_rate']:.1%}")
df.head()

In [ ]:
print("Crime types:")
print(df["crime_type"].value_counts(dropna=False))
print("\nPeríodo:")
print(df["period"].value_counts(dropna=False))

## 3. Persistir cache em parquet

Parquet é ~5x mais rápido que xlsx e cabe no `.gitignore` (não versionamos).
Notebooks subsequentes podem ler direto desse cache.

In [ ]:
parquet_path = exporter.write_processed_parquet(df)
print(f"Salvo em: {parquet_path.relative_to(config.ROOT_DIR)}")
print(f"Tamanho:  {parquet_path.stat().st_size / 1024:.1f} KB")

## 4. Gerar `summary.json`

Esse arquivo vai para o frontend (painel de estatísticas e badges
de "hot zones") e também ao Supabase futuramente.

In [ ]:
summary = cleaner.summarize(df)
summary_path = exporter.write_summary(summary)
print(f"Salvo em: {summary_path.relative_to(config.ROOT_DIR)}")
summary